In [0]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats
from sklearn.preprocessing import StandardScaler


In [0]:
df = pd.read_csv("11_Baseketball_Team_Performance_Analysis.csv")
df.head()
df.shape

# Checking null values

In [0]:
df.isnull().sum()

# Filling null values with median

In [0]:
numeric_cols = df.select_dtypes(include='number').columns

for col in numeric_cols:
    df[col] = df[col].fillna(df[col].median())

# Checking if nulls are gone
print("Nulls remaining:", df.isnull().sum().sum())

# Checking for duplicates

In [0]:
duplicates = df.duplicated().sum()
print(f"Duplicate rows: {duplicates}")

if duplicates > 0:
    df = df.drop_duplicates()
    print(f"Duplicates removed. New shape: {df.shape}")

# Outlier Detection

In [0]:
# Box plots for outlier detection
features_to_check = ['offensive_rating', 'defensive_rating', 'net_rating', 'points', 'turnovers', '3_pointer_attempts']

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for i, col in enumerate(features_to_check):
    sns.boxplot(x=df[col], ax=axes[i])
    axes[i].set_title(f'{col.replace("_", " ").title()} - Outliers')

plt.tight_layout()
plt.show()

# Outlier Handling (IQR Method)

In [0]:
print("\nOutlier Handling:")
for col in numeric_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    outliers = ((df[col] < lower_bound) | (df[col] > upper_bound)).sum()
    if outliers > 0:
        print(f"{col}: {outliers} outliers detected")
        # Cap outliers instead of removing (preserves data size)
        df[col] = df[col].clip(lower_bound, upper_bound)

In [0]:
# Scatter plots - key metrics vs win percentage
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Points vs Win %
axes[0].scatter(df['points'], df['win_percentage'], alpha=0.3)
axes[0].set_xlabel('Points')
axes[0].set_ylabel('Win Percentage')
axes[0].set_title('Points vs Win Percentage')

# Net Rating vs Win %
axes[1].scatter(df['net_rating'], df['win_percentage'], alpha=0.3, color='green')
axes[1].set_xlabel('Net Rating')
axes[1].set_ylabel('Win Percentage')
axes[1].set_title('Net Rating vs Win Percentage')

plt.tight_layout()
plt.show()

In [0]:
# Correlation heatmap
plt.figure(figsize=(20, 16))
corr = df[numeric_cols].corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm')
plt.title('Correlation Heatmap - All Features')
plt.tight_layout()
plt.show()

# Feature Engineering

In [0]:
# 1. point difference = points scored - opponent points
df['point_diff'] = df['points'] - df['opponent_points']

# 2. assist to turnover ratio (safe division)
df['assist_turnover_ratio'] = np.where(df['turnovers'] == 0, 0, df['assists'] / df['turnovers'])

# 3. home win percentage (handle division by zero)
df['home_games'] = df['home_wins'] + df['home_losses']
df['home_win_pct'] = np.where(df['home_games'] == 0, 0.5, df['home_wins'] / df['home_games'])

# 4. away win percentage
df['away_games'] = df['away_wins'] + df['away_losses']
df['away_win_pct'] = np.where(df['away_games'] == 0, 0.5, df['away_wins'] / df['away_games'])

# 5. home court advantage
df['home_advantage'] = df['home_win_pct'] - df['away_win_pct']

# 6. conference win percentage
df['conf_games'] = df['conference_wins'] + df['conference_losses']
df['conf_win_pct'] = np.where(df['conf_games'] == 0, 0.5, df['conference_wins'] / df['conf_games'])

# 7. scoring efficiency
df['scoring_efficiency'] = np.where(df['field_goal_attempts'] == 0, 0, df['points'] / df['field_goal_attempts'])

# 8. defensive pressure
df['defensive_pressure'] = df['steals'] + df['blocks']

print("New features created!")

# Feature Correlations with Win %

In [0]:
new_cols = ['point_diff', 'assist_turnover_ratio', 'home_advantage', 'conf_win_pct', 'scoring_efficiency', 'defensive_pressure']

for col in new_cols:
    corr_val = df[col].corr(df['win_percentage'])
    print(f"{col} -> correlation with win%: {round(corr_val, 3):.3f}")

# Clean Up Helper Columns

In [0]:
# Drop helper columns used for calculations
helper_cols = ['home_games', 'away_games', 'conf_games']
df = df.drop(columns=helper_cols)

print(f"Dropped {len(helper_cols)} helper columns")
print(f"New shape: {df.shape}")

# Normalization

In [0]:
scaler = StandardScaler()

# Get all numeric columns after feature engineering
all_numeric_cols = df.select_dtypes(include='number').columns.tolist()
features_to_scale = [col for col in all_numeric_cols if col != 'win_percentage']

# Fit and scale the features
df_scaled = df.copy()
df_scaled[features_to_scale] = scaler.fit_transform(df[features_to_scale])

print("Normalization completed!")
print("\nBefore scaling - Points (first 5):", df['points'].head().values)
print("After scaling - Points (first 5):", df_scaled['points'].head().values)
print("\nScaled data shape:", df_scaled.shape)

# Final Dataset

In [0]:
print("Final shape:", df.shape)
df.head()

# HYPOTHESIS TESTING
## Test 1: High Points lead to Higher Win Percentage
**Test**: Independent t-test (split by median points)

In [0]:
# Hypothesis Test 1: High Points vs Win Percentage

# Split by median points
median_points = df['points'].median()
print(f"Median points: {median_points:.1f}")

high_points = df[df['points'] > median_points]['win_percentage']
low_points = df[df['points'] <= median_points]['win_percentage']

print(f"High points teams (n={len(high_points)}): mean win% = {high_points.mean():.3f}")
print(f"Low points teams (n={len(low_points)}): mean win% = {low_points.mean():.3f}")

# t-test
t_stat, p_value = stats.ttest_ind(high_points, low_points)
print(f"\nT-test Results:")
print(f"t-statistic = {t_stat:.3f}")
print(f"p-value = {p_value:.6f}")

if p_value < 0.05:
    print("REJECT H0: High points teams have significantly higher win % (p < 0.05)")
else:
    print("Fail to reject H0")

# Create points_group column
df['points_group'] = df['points'] > median_points
df['points_group'] = df['points_group'].map({True: 'High', False: 'Low'})

# Visualization
plt.figure(figsize=(10, 4))

plt.subplot(1, 2, 1)
sns.boxplot(data=df, x='points_group', y='win_percentage')
plt.title('Win % by Points Group')

plt.subplot(1, 2, 2)
plt.scatter(df['points'], df['win_percentage'], alpha=0.5)
plt.axvline(median_points, color='red', linestyle='--', label=f'Median: {median_points:.1f}')
plt.xlabel('Points')
plt.ylabel('Win %')
plt.title('Points vs Win %')
plt.legend()

plt.tight_layout()
plt.show()